# KPIs

## 1.Monthly Consolidated Revenue

In [0]:
%sql
SELECT c.company_id, c.company_name, year(posting_date) AS year, month(posting_date) AS month, SUM(credit_amount) AS total_revenue
FROM pl_workforce_catalog.gold.fact_general_ledgers f
JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
JOIN pl_workforce_catalog.gold.dim_company c ON f.company_id = c.company_id
WHERE a.account_type = 'Revenue'
GROUP BY c.company_id, c.company_name, year(posting_date), month(posting_date)
ORDER BY c.company_id, c.company_name, year, month;

## 2.Cost of Sales by Month 

In [0]:
%sql
SELECT c.company_id,c.company_name, year(posting_date) AS year,month(posting_date) AS month,SUM(debit_amount) AS cost_of_sales
FROM pl_workforce_catalog.gold.fact_general_ledgers f
JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
JOIN pl_workforce_catalog.gold.dim_company c ON f.company_id = c.company_id
WHERE a.category = 'Operating Expense'
GROUP BY c.company_id,year(posting_date), month(posting_date),company_name
ORDER BY c.company_id,year(posting_date), month(posting_date),company_name;
                

## 3.Monthly Gross Profit Margin

In [0]:
%sql
SELECT r.year, r.month, r.revenue, c.cost,
       (r.revenue - c.cost) AS gross_profit,
       ROUND((r.revenue - c.cost) / r.revenue * 100, 2) AS gross_margin_pct
FROM (
    SELECT year(posting_date) AS year,
           month(posting_date) AS month,
           SUM(credit_amount) AS revenue
    FROM pl_workforce_catalog.gold.fact_general_ledgers f
    JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
    WHERE a.account_type = 'Revenue'
    GROUP BY year(posting_date), month(posting_date)
) r
JOIN (
    SELECT year(posting_date) AS year,
           month(posting_date) AS month,
           SUM(debit_amount) AS cost
    FROM pl_workforce_catalog.gold.fact_general_ledgers f
    JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
    WHERE a.category = 'Operating Expense'
    GROUP BY year(posting_date), month(posting_date)
) c
ON r.year = c.year AND r.month = c.month
ORDER BY r.year, r.month;

## 4.Operating Expense Breakdown

In [0]:
%sql
SELECT f.company_id, c.company_name, a.category, SUM(f.debit_amount) AS total_expense
FROM pl_workforce_catalog.gold.fact_general_ledgers f
JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
JOIN pl_workforce_catalog.gold.dim_company c ON f.company_id = c.company_id
WHERE a.account_type != 'Revenue'
GROUP BY f.company_id, c.company_name, a.category
ORDER BY f.company_id, c.company_name, a.category;

## 5.Average Compensation by Position

In [0]:
%sql
SELECT e.position, f.company_id, AVG(f.gross_salary + f.bonus + f.overtime_pay + f.commission) AS avg_compensation
FROM pl_workforce_catalog.gold.fact_payroll f
JOIN pl_workforce_catalog.gold.dim_employee e ON f.employee_id = e.employee_id
GROUP BY e.position, f.company_id
ORDER BY f.company_id, e.position;

## 6.Net Profit by Month

In [0]:
%sql
SELECT year(posting_date) AS year,
       month(posting_date) AS month,
       SUM(CASE WHEN a.account_type = 'Revenue' THEN credit_amount ELSE 0 END) -
       SUM(CASE WHEN a.account_type != 'Revenue' THEN debit_amount ELSE 0 END) AS net_profit
FROM pl_workforce_catalog.gold.fact_general_ledgers f
JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
GROUP BY year(posting_date), month(posting_date);

## 7.Overtime and Bonus Analysis

In [0]:
%sql
SELECT p.department_id,
       d.department_name,
       SUM(p.overtime_pay) AS total_overtime,
       SUM(p.bonus) AS total_bonus,
       SUM(p.overtime_pay + p.bonus + p.commission) AS total_variable_comp,
       ROUND(SUM(p.overtime_pay) / SUM(p.gross_salary), 2) AS overtime_to_base_salary_ratio
FROM pl_workforce_catalog.gold.fact_payroll p
JOIN pl_workforce_catalog.gold.dim_department d ON p.department_id = d.department_id
GROUP BY p.department_id, d.department_name
ORDER BY p.department_id;

## 8.Cost per Department 

In [0]:
%sql
SELECT department_id,SUM(gross_salary + bonus + overtime_pay + commission) AS total_cost
FROM pl_workforce_catalog.gold.fact_payroll
GROUP BY department_id;
        

## 9.Headcount Distribution by Department

In [0]:
%sql
SELECT department_id, COUNT(employee_id) AS active_headcount
FROM pl_workforce_catalog.gold.dim_employee
WHERE is_active = 'TRUE'
GROUP BY department_id;

## 10.Payroll Cost as % of Company Revenue

In [0]:
%sql
SELECT year(p.pay_date) AS year,
       month(p.pay_date) AS month,
       SUM(p.gross_salary) AS payroll_cost,
       r.total_revenue,
       ROUND(SUM(p.gross_salary) / r.total_revenue * 100, 2) AS payroll_pct,
       ROUND(r.total_revenue / SUM(p.gross_salary), 2) AS labor_efficiency_ratio
FROM pl_workforce_catalog.gold.fact_payroll p
JOIN (
    SELECT year(posting_date) AS year,
           month(posting_date) AS month,
           SUM(credit_amount) AS total_revenue
    FROM pl_workforce_catalog.gold.fact_general_ledgers f
    JOIN pl_workforce_catalog.gold.dim_account a ON f.account_id = a.account_id
    WHERE a.account_type = 'Revenue'
    GROUP BY year(posting_date), month(posting_date)
) r
ON year(p.pay_date) = r.year AND month(p.pay_date) = r.month
GROUP BY year(p.pay_date), month(p.pay_date), r.total_revenue;